# Import libraries 

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_AUTO_SHAPE_TYPE, MSO_CONNECTOR
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt
# Basic figure setting
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# Unified color palette
COLOR_DEMAND = "#2E86AB" # blue for demand trend
COLOR_BAR = "#58A55C" # green for bar charts
COLOR_SCATTER = "#8E44AD" # purple for scatter plots
COLOR_ANOMALY = "#C0392B" # red for anomalies
COLOR_BEFORE = "#AAB7B8" # grey for before-cleaning data
COLOR_MODEL = "#E67E22" # orange for model result
COLOR_ACTUAL = "#2C3E50" # dark color for actual values
COLOR_RF = "#16A085" # green-blue for Random Forest
COLOR_XGB = "#D35400" # orange-red for XGBoost

# Read the datasets

In [2]:
def Read_CSV_file(file_location):
    df = pd.read_csv(file_location)
    return df

energy_df_location = "Datasets/energy_dataset.csv"
weather_df_location = "Datasets/weather_features.csv"
##############################################################


def Shape_dataframe(energy_df,weather_df):
    print("Energy data shape:", energy_df.shape)
    display(energy_df.head())

    print("Weather data shape:", weather_df.shape)
    display(weather_df.head())




def show_columns(df):
    cols = df.columns.tolist()

    print("Columns:")
    print(cols)

    print("\nTotal columns:", len(cols))



def Show_missing_values(df):
    print("Missing values in data:")
    display(df.isnull().sum().sort_values(ascending=False).head(50))
###############################################################################
energy_df = Read_CSV_file(energy_df_location )
weather_df = Read_CSV_file(weather_df_location)
#show_columns(energy_df)
#show_columns(weather_df)
#Shape_dataframe(energy_df,weather_df)
#Show_missing_values(energy_df)
#print("------------------------------------------------")
#Show_missing_values(weather_df)


In [3]:
# Convert time columns to datetime 
def convert_time_columns_datetime(energy_df, weather_df):
    energy_copy2 = energy_df.copy()
    weather_copy2 = weather_df.copy()
    energy_copy2["time"] = pd.to_datetime( energy_copy2["time"], utc=True  )
    weather_copy2["dt_iso"] = pd.to_datetime( weather_copy2["dt_iso"], utc=True)
    return energy_copy2, weather_copy2
    
# Remove timezone information
def remove_timezone(energy_df, weather_df):
    energy_copy = energy_df.copy()
    weather_copy = weather_df.copy()

    energy_copy["time"] = energy_copy["time"].dt.tz_convert(None)
    weather_copy["dt_iso"] = weather_copy["dt_iso"].dt.tz_convert(None)
    return energy_copy, weather_copy
    

def rename_and_remove_duplicates(energy_df, weather_df): 
    energy = energy_df.copy()
    weather = weather_df.copy()

    # Rename weather timestamp column
    weather = weather.rename(columns={"dt_iso": "time"})

    # Remove duplicate rows
    energy = energy.drop_duplicates()
    weather = weather.drop_duplicates()

    return energy, weather

def remove_empty_columns(energy_df, weather_df):
    energy = energy_df.copy()
    weather = weather_df.copy()
    energy = energy.dropna(axis=1, how="all")
    weather = weather.dropna(axis=1, how="all")
    return energy, weather

def Preprocess_data(energy_df, weather_df):    
    # Step 1: Convert time columns to datetime
    energy, weather = convert_time_columns_datetime( energy_df, weather_df )
    
    # Step 2: Remove timezone
    energy, weather = remove_timezone( energy, weather )
    
    # Step 3: Remove completely empty columns
    energy, weather = remove_empty_columns(energy, weather )
    
    # Step 4: Rename time column and remove duplicates
    energy, weather = rename_and_remove_duplicates(energy, weather )
    
    return energy, weather
########################################################################################
# Usage
energy_clean, weather_clean = Preprocess_data(energy_df, weather_df )
#energy_clean

In [4]:
import numpy as np

def get_averages_weather(weather_df):
    weather_df = weather_df.copy()
    weather_numeric_cols = weather_df.select_dtypes(include=[np.number]).columns.tolist()
    weather_avg = weather_df.groupby("time")[weather_numeric_cols].mean().reset_index()
    #print(weather_numeric_cols)
    return weather_avg

weather_avg = get_averages_weather(weather_clean)


def Join_datasests(energy_clean, weather_avg):
    df = pd.merge(energy_clean, weather_avg, on="time", how="inner") 
    # Sort data by time
    df_combine = df.sort_values("time").reset_index(drop=True)
    df_combine = df_combine.set_index("time")
    return df_combine 
    
combine_df = Join_datasests(energy_clean, weather_avg)
combine_df

,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage consumption,...,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id
time,,,,,,,,,,,,,,,,,,,,,
2014-12-31 23:00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,863.0,...,272.491463,1016.4,82.4,2.0,135.2,0.0,0.0,0.0,0.0,800.0
2015-01-01 00:00:00,449.0,328.0,0.0,5196.0,4755.0,158.0,0.0,0.0,0.0,920.0,...,272.512700,1016.2,82.4,2.0,135.8,0.0,0.0,0.0,0.0,800.0
2015-01-01 01:00:00,448.0,323.0,0.0,4857.0,4581.0,157.0,0.0,0.0,0.0,1164.0,...,272.099137,1016.8,82.0,2.4,119.0,0.0,0.0,0.0,0.0,800.0
2015-01-01 02:00:00,438.0,254.0,0.0,4314.0,4131.0,160.0,0.0,0.0,0.0,1503.0,...,272.089469,1016.6,82.0,2.4,119.2,0.0,0.0,0.0,0.0,800.0
2015-01-01 03:00:00,428.0,187.0,0.0,4130.0,3840.0,156.0,0.0,0.0,0.0,1826.0,...,272.145900,1016.6,82.0,2.4,118.4,0.0,0.0,0.0,0.0,800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-12-31 18:00:00,297.0,0.0,0.0,7634.0,2628.0,178.0,0.0,0.0,0.0,1.0,...,285.550000,1029.2,73.6,1.4,151.4,0.0,0.0,0.0,0.0,800.0
2018-12-31 19:00:00,296.0,0.0,0.0,7241.0,2566.0,174.0,0.0,0.0,0.0,1.0,...,283.350000,1030.0,68.0,1.8,160.4,0.0,0.0,0.0,0.0,800.0
2018-12-31 20:00:00,292.0,0.0,0.0,7025.0,2422.0,168.0,0.0,0.0,0.0,50.0,...,281.350000,1029.8,71.4,2.6,226.0,0.0,0.0,0.0,0.0,800.0


In [9]:
import pickle
import os
def dump_energy_weather_df(df):
    filepath = "pickle_files/energy_weather_df"
    with open(filepath, "wb") as file:
        pickle.dump(df, file)
        print(f"Saved successfully: {filepath}")

def load_energy_weather_df():
    filepath = "pickle_files/energy_weather_df"
    with open(filepath, "rb") as file:
        df = pickle.load(file)

    print(f"Loaded successfully: {filepath}")
    return df
#########################################################################

# Save
#dump_energy_weather_df(combine_df)
# Load
combine_df = load_energy_weather_df()


Loaded successfully: pickle_files/energy_weather_df


In [13]:
print(combine_df.columns)

Index(['generation biomass', 'generation fossil brown coal/lignite',
       'generation fossil coal-derived gas', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation fossil oil shale', 'generation fossil peat',
       'generation geothermal', 'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation marine',
       'generation nuclear', 'generation other', 'generation other renewable',
       'generation solar', 'generation waste', 'generation wind offshore',
       'generation wind onshore', 'forecast solar day ahead',
       'forecast wind onshore day ahead', 'total load forecast',
       'total load actual', 'price day ahead', 'price actual', 'temp',
       'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed',
       'wind_deg', 'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all',
       'weather_id'],
      dtype='str')


#  Anomailies detections 

In [19]:
import pandas as pd
import numpy as np

############################################################################
# STEP 1: Clean pressure anomalies
############################################################################

def clean_pressure_anomalies(df):

    df_copy = df.copy()

    if "pressure" in df_copy.columns:

        df_copy["pressure"] = pd.to_numeric(
            df_copy["pressure"],
            errors="coerce"
        )

        pressure_anomaly = (
            (df_copy["pressure"] < 900) |
            (df_copy["pressure"] > 1100)
        )

        df_copy.loc[pressure_anomaly, "pressure"] = np.nan

    return df_copy


############################################################################
# STEP 2: Interpolate missing values
############################################################################

def interpolate_missing_values(df):

    df_copy = df.copy()

    # Ensure index is datetime
    df_copy.index = pd.to_datetime(df_copy.index)

    numeric_cols = df_copy.select_dtypes(
        include=[np.number]
    ).columns

    df_copy[numeric_cols] = (
        df_copy[numeric_cols]
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    return df_copy


############################################################################
# STEP 3: Pressure summary
############################################################################

def get_pressure_summary(df):

    if "pressure" not in df.columns:
        return None

    return {
        "pressure_min": df["pressure"].min(),
        "pressure_max": df["pressure"].max(),
        "pressure_mean": df["pressure"].mean(),
        "missing_pressure": df["pressure"].isnull().sum()
    }


############################################################################
# STEP 4: Complete cleaning pipeline
############################################################################

def clean_weather_dataset(df):

    df_clean = clean_pressure_anomalies(df)
    df_clean = interpolate_missing_values(df_clean)

    return df_clean


############################################################################
# STEP 5: Temperature conversion
############################################################################

def convert_temperature(df):

    df_copy = df.copy()

    if "temp" in df_copy.columns:

        if df_copy["temp"].mean() > 100:
            df_copy["temp_c"] = df_copy["temp"] - 273.15
        else:
            df_copy["temp_c"] = df_copy["temp"]

    else:
        df_copy["temp_c"] = np.nan

    return df_copy


############################################################################
# EXECUTION
############################################################################

clean_combine_df = clean_weather_dataset(combine_df)

pressure_summary = get_pressure_summary(clean_combine_df)

clean_combine_df = convert_temperature(clean_combine_df)

print("Pressure Summary:")
print(pressure_summary)

print(clean_combine_df[["total load actual", "temp_c"]].head())

Pressure Summary:
{'pressure_min': np.float64(933.2), 'pressure_max': np.float64(1039.8), 'pressure_mean': np.float64(1016.2181490240973), 'missing_pressure': np.int64(0)}
                     total load actual    temp_c
time                                            
2014-12-31 23:00:00            25385.0 -0.658537
2015-01-01 00:00:00            24382.0 -0.637300
2015-01-01 01:00:00            22734.0 -1.050862
2015-01-01 02:00:00            21286.0 -1.060531
2015-01-01 03:00:00            20264.0 -1.004100
